# HEOR Single-Task Baselines

Colab-first downstream modeling notebook following the R0/Manuscript3 layout: explicit setup, data preparation, split checks, model-specific training sections, saved predictions, and summary tables.

# 1. Import Libraries, Configuration Setup, and Load Dataset

## 1.1 Install and import libraries

In [ ]:
# Notebook / package setup
import importlib.util
import subprocess
import sys

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "joblib": "joblib",
    "lightgbm": "lightgbm",
    "sklearn": "scikit-learn",
    "torch": "torch",
    "transformers": "transformers",
    "tqdm": "tqdm",
}

missing = [
    pip_name
    for module_name, pip_name in required.items()
    if importlib.util.find_spec(module_name) is None
]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import gc
import json
import os
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")


## 1.2 Seeds and device

In [ ]:
SEED = 42
RANDOM_STATE = SEED


def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_FP16 = torch.cuda.is_available()

print("Device:", DEVICE)
print("Seed:", SEED)
print("FP16 available:", USE_FP16)


## 1.3 Configuration

In [ ]:
# Portable project-path setup.
# Override any of these by exporting the matching env var before launching
# the notebook (e.g. `export PROJECT_ROOT=/path/to/VulnerableCancerPatients`):
#   PROJECT_ROOT  - root of the experiment tree
#   SCRIPTS_DIR   - shared helper modules (default: PROJECT_ROOT/scripts)
#   DATA_DIR      - shared data directory (default: PROJECT_ROOT/data)
import os
import sys
from pathlib import Path

FOLDER_NAME = "05_HEOR_SingleTaskBaselines"
COLAB_DEFAULT = Path("/content/drive/MyDrive/NLP_Projects/VulnerableCancerPatients")


def _resolve_project_root() -> Path:
    env = os.environ.get("PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    try:
        from google.colab import drive  # type: ignore

        drive.mount("/content/drive", force_remount=False)
        if COLAB_DEFAULT.exists():
            return COLAB_DEFAULT.resolve()
    except ImportError:
        pass
    cwd = Path.cwd().resolve()
    if cwd.name == FOLDER_NAME:
        return cwd.parent
    if (cwd / FOLDER_NAME).exists():
        return cwd
    return cwd


PROJECT_ROOT = _resolve_project_root()
BASE_DIR = PROJECT_ROOT / FOLDER_NAME if (PROJECT_ROOT / FOLDER_NAME).exists() else PROJECT_ROOT
SCRIPTS_DIR = Path(os.environ.get("SCRIPTS_DIR", PROJECT_ROOT / "scripts")).expanduser().resolve()
DATA_DIR = Path(os.environ.get("DATA_DIR", PROJECT_ROOT / "data")).expanduser().resolve()
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

print("Project root:", PROJECT_ROOT)
print("Base:", BASE_DIR)
print("Scripts:", SCRIPTS_DIR, "(exists)" if SCRIPTS_DIR.exists() else "(MISSING)")
print("Data:", DATA_DIR, "(exists)" if DATA_DIR.exists() else "(MISSING)")
print("Outputs:", OUTPUT_DIR)


## 1.4 Mount Drive or use local project folder

In [ ]:
# Path setup consolidated into the portable block above.


## 1.5 Load shared data and split

In [ ]:
import sys
import os
import importlib.util

# Filter out any existing 'scripts' related paths from sys.path that might be interfering
new_sys_path = [p for p in sys.path if not (isinstance(p, str) and p.startswith('/content/drive/MyDrive/') and 'scripts' in p and p != str(SCRIPTS_DIR))]

# Ensure the correct SCRIPTS_DIR is at the very front of sys.path
# Remove if already present to ensure it's re-inserted at the beginning
if str(SCRIPTS_DIR) in new_sys_path:
    new_sys_path.remove(str(SCRIPTS_DIR))
new_sys_path.insert(0, str(SCRIPTS_DIR))
sys.path = new_sys_path

print("Current sys.path (after cleanup and insertion):")
for p in sys.path:
    print(p)

try:
    # --- Explicitly load data_utils module ---
    data_utils_path = SCRIPTS_DIR / "data_utils.py"
    print(f"Checking for data_utils.py at {data_utils_path}: {data_utils_path.exists()}")
    if not data_utils_path.exists():
        raise FileNotFoundError(f"Expected module file not found: {data_utils_path}")
    spec_data_utils = importlib.util.spec_from_file_location("data_utils", data_utils_path)
    if spec_data_utils is None:
        raise ImportError(f"Could not get module spec for data_utils at {data_utils_path}.")
    data_utils_module = importlib.util.module_from_spec(spec_data_utils)
    sys.modules["data_utils"] = data_utils_module
    spec_data_utils.loader.exec_module(data_utils_module)
    print("Successfully loaded data_utils via importlib.util.")

    # --- Explicitly load hp_search module ---
    hp_search_path = SCRIPTS_DIR / "hp_search.py"
    print(f"Checking for hp_search.py at {hp_search_path}: {hp_search_path.exists()}")
    if not hp_search_path.exists():
        raise FileNotFoundError(f"Expected module file not found: {hp_search_path}")
    spec_hp_search = importlib.util.spec_from_file_location("hp_search", hp_search_path)
    if spec_hp_search is None:
        raise ImportError(f"Could not get module spec for hp_search at {hp_search_path}.")
    hp_search_module = importlib.util.module_from_spec(spec_hp_search)
    sys.modules["hp_search"] = hp_search_module
    spec_hp_search.loader.exec_module(hp_search_module)
    print("Successfully loaded hp_search via importlib.util.")

    # --- Explicitly load metrics module ---
    metrics_path = SCRIPTS_DIR / "metrics.py"
    print(f"Checking for metrics.py at {metrics_path}: {metrics_path.exists()}")
    if not metrics_path.exists():
        raise FileNotFoundError(f"Expected module file not found: {metrics_path}")
    spec_metrics = importlib.util.spec_from_file_location("metrics", metrics_path)
    if spec_metrics is None:
        raise ImportError(f"Could not get module spec for metrics at {metrics_path}.")
    metrics_module = importlib.util.module_from_spec(spec_metrics)
    sys.modules["metrics"] = metrics_module
    spec_metrics.loader.exec_module(metrics_module)
    print("Successfully loaded metrics via importlib.util.")

    # --- Explicitly load train_eval module ---
    train_eval_path = SCRIPTS_DIR / "train_eval.py"
    print(f"Checking for train_eval.py at {train_eval_path}: {train_eval_path.exists()}")
    if not train_eval_path.exists():
        raise FileNotFoundError(f"Expected module file not found: {train_eval_path}")
    spec_train_eval = importlib.util.spec_from_file_location("train_eval", train_eval_path)
    if spec_train_eval is None:
        raise ImportError(f"Could not get module spec for train_eval at {train_eval_path}.")
    train_eval_module = importlib.util.module_from_spec(spec_train_eval)
    sys.modules["train_eval"] = train_eval_module
    spec_train_eval.loader.exec_module(train_eval_module)
    print("Successfully loaded train_eval via importlib.util.")

    # Now, import components from the explicitly loaded modules
    from data_utils import (
        CANCER_LABELS,
        EMOTION_LABELS_3,
        EMOTION_PROB_COLS_3,
        HEOR_SUBSCALES,
        ROLE_LABELS,
        prepare_annotation_frame,
    )
    from metrics import DEFAULT_BOOTSTRAP_N
    from train_eval import TrainSettings, run_classifier_search, run_heor_r0_mtl_search

    print("Successfully imported components from data_utils, metrics, and train_eval.")

    df = prepare_annotation_frame(ANNOTATION_PATH, SPLIT_PATH)
    train_df = df[df["split"] == "train"].copy()
    val_df = df[df["split"] == "val"].copy()
    test_df = df[df["split"] == "test"].copy()

    print("Prepared shape:", df.shape)
    print(df["split"].value_counts().sort_index())
    print("Bootstrap helper default:", DEFAULT_BOOTSTRAP_N)
    display(df[["source_row", "split", "human_emotion_3class", "llm_argmax_3class", "ai_high_need_flag"]].head())

except Exception as e:
    print(f"An error occurred during module loading or subsequent imports: {type(e).__name__}: {e}")
    # Re-raise the exception to make sure the user sees the full traceback if it's not ModuleNotFoundError
    raise

In [ ]:
import os

print(f"Contents of {SCRIPTS_DIR}:")
if SCRIPTS_DIR.exists():
    for item in os.listdir(SCRIPTS_DIR):
        print(item)
else:
    print(f"The directory {SCRIPTS_DIR} does not exist.")

# 2. Data Exploration and Master Data Preparation

## 2.1 Master data check

In [ ]:
print("Columns:", len(df.columns))
print("Rows:", len(df))
print()
print("Human 3-class distribution:")
print(df["human_emotion_3class"].value_counts(dropna=False))
print()
print("LLM argmax 3-class distribution:")
print(df["llm_argmax_3class"].value_counts(dropna=False))
print()
print("HEOR subscale summaries:")
display(df[HEOR_SUBSCALES].describe().T)


## 2.2 Canonical split check

In [ ]:
split_summary = (
    df.groupby("split")
    .agg(
        n=("source_row", "size"),
        source_rows=("source_row", "nunique"),
        human_negative=("human_emotion_3class", lambda s: int((s == "NEGATIVE").sum())),
        human_neutral=("human_emotion_3class", lambda s: int((s == "NEUTRAL").sum())),
        human_positive=("human_emotion_3class", lambda s: int((s == "POSITIVE").sum())),
    )
    .reset_index()
)
display(split_summary)

assert set(df["split"].unique()) == {"train", "val", "test"}
assert df["source_row"].is_unique, "Expected one prepared row per source row."

if "posts" in df.columns:
    repeated_text_splits = df.groupby("posts")["split"].nunique()
    n_repeated_text_leaks = int((repeated_text_splits > 1).sum())
    print("Repeated exact-text groups crossing splits:", n_repeated_text_leaks)

master_split_path = OUTPUT_DIR / "master_prepared_split.csv"
df.to_csv(master_split_path, index=False)
print("Saved prepared split:", master_split_path)


## 2.3 Shared helper functions

In [ ]:
def maybe_smoke_sample(frame: pd.DataFrame, per_split: int = SMOKE_PER_SPLIT) -> pd.DataFrame:
    if not SMOKE_TEST:
        return frame.copy()
    return (
        frame.groupby("split", group_keys=False)
        .apply(lambda x: x.sample(min(len(x), per_split), random_state=SEED))
        .reset_index(drop=True)
    )


def make_train_settings(model_name: str = ALBERT_MODEL_NAME, batch_size: int = BATCH_SIZE, n_iter: int = N_ITERATIONS):
    return TrainSettings(
        model_name=model_name,
        max_length=MAX_TOKEN_LENGTH,
        batch_size=batch_size,
        patience=PATIENCE,
        n_iter=n_iter,
        bootstrap_n=N_BOOT,
        seed=SEED,
    )


def append_metric(metrics_list, metric_row):
    metrics_list.append(metric_row)
    display(pd.DataFrame(metrics_list))


df_run = maybe_smoke_sample(df)
print("Run shape:", df_run.shape)
print(df_run["split"].value_counts().sort_index())


# 3. Independent HEOR Single-Task Models

## 3.1 Select subscales

In [ ]:
subscales_to_run = HEOR_SUBSCALES if RUN_ALL_SUBSCALES else MINIMUM_REVIEWER_SUBSCALES
print("Subscales to run:", subscales_to_run)
model_metrics = []


## 3.2 Train one ALBERT classifier per subscale

In [ ]:
metrics_path = os.path.join(OUTPUT_DIR, "heor_single_task_metrics.csv")
print({metrics_path})


In [ ]:
import os
import pandas as pd
from pandas.errors import EmptyDataError

metrics_path = os.path.join(OUTPUT_DIR, "heor_single_task_metrics.csv")
print({metrics_path})

settings = make_train_settings()

for subscale in subscales_to_run:
    condition = f"single_task_{subscale}"

    if os.path.exists(metrics_path):
        print(f"Reading existing metrics from {metrics_path}")
        try:
            existing_metrics = pd.read_csv(metrics_path)

            if "condition_name" in existing_metrics.columns:
                completed_conditions = set(existing_metrics["condition_name"].astype(str))
            else:
                completed_conditions = set()
        except EmptyDataError:
            existing_metrics = pd.DataFrame()
            completed_conditions = set()
    else:
        print(f"Metrics file {metrics_path} not found. Creating a new DataFrame.")
        existing_metrics = pd.DataFrame()
        completed_conditions = set()

    if condition in completed_conditions:
        print(f"Skipping {condition}, already completed.")
        continue

    print(f"\n===== Running {condition} =====")

    metric_row, _ = run_classifier_search(
        df=df_run,
        output_dir=OUTPUT_DIR,
        condition_name=condition,
        text_col="text_regular",
        hard_label_col=subscale,
        soft_label_cols=None,
        loss_type="hard",
        target_source=f"llm_heor_{subscale}",
        settings=settings,
        class_names=["0", "1", "2", "3"],
        num_labels=4,
        eval_label_col=subscale,
        selection_metric="weighted_f1",
        include_ordinal=True,
    )

    # 确保这一行里有 condition_name，方便下次跳过
    metric_row["condition_name"] = condition

    current_row = pd.DataFrame([metric_row])
    updated_metrics = pd.concat([existing_metrics, current_row], ignore_index=True)

    # 防止意外重复
    updated_metrics = updated_metrics.drop_duplicates(
        subset=["condition_name"],
        keep="last"
    )

    updated_metrics.to_csv(metrics_path, index=False)

    print(f"Saved {condition} result to {metrics_path}.")